# Kaggle住宅価格予測（Python / 講義演習の整理版）
授業で作成したコードを、公開用に個人情報と実行結果を除去してまとめたものです。
Kaggle提供の train.csv / test.csv が必要です。
モデルの検証スコアや実行再現性は未確認です。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_ID = train_df["Id"]
test_ID = test_df["Id"]
y = np.log1p(train_df["SalePrice"])
data = pd.concat([train_df.drop("Id", axis=1), test_df.drop("Id", axis=1)], axis=0)

In [ ]:
missing_count = data.isnull().sum()
missing_ratio = data.isnull().mean() * 100
missing_df = pd.DataFrame({"欠損数": missing_count, "欠損率(%)": missing_ratio})
missing_df = missing_df[missing_df["欠損数"] > 0]
print(missing_df)

In [ ]:
numeric_cols = data.select_dtypes(include=["number"]).columns
missing_ratio = data[numeric_cols].isnull().mean()
cols_to_fill = missing_ratio[(missing_ratio > 0) & (missing_ratio < 0.3)].index
if "SalePrice" in cols_to_fill:
    cols_to_fill = cols_to_fill.drop("SalePrice")
data[cols_to_fill] = data[cols_to_fill].fillna(data[cols_to_fill].median())
for col in data.columns:
    if col != "SalePrice" and data[col].dtype == "object":
        data[col] = data[col].fillna("None")

In [ ]:
for col in data.select_dtypes(include="object").columns:
    data[col] = LabelEncoder().fit_transform(data[col])

In [ ]:
data["TotalSF"] = data["TotalBsmtSF"] + data["1stFlrSF"] + data["2ndFlrSF"]

In [ ]:
train = data[pd.notna(data.SalePrice)]
test = data[pd.isna(data.SalePrice)].drop(columns=["SalePrice"])

In [ ]:
lasso = Lasso(alpha=0.0005)
lgb_model = lgb.LGBMRegressor(learning_rate=0.05, n_estimators=1000)
xgb_model = xgb.XGBRegressor(learning_rate=0.05, n_estimators=1000)

In [ ]:
train = train.drop(columns=["SalePrice"])
lasso.fit(train, y)
lgb_model.fit(train, y)
xgb_model.fit(train, y)

In [ ]:
pred_lasso = np.expm1(lasso.predict(test))
pred_lgb = np.expm1(lgb_model.predict(test))
pred_xgb = np.expm1(xgb_model.predict(test))
pred_final = (pred_lasso + pred_lgb + pred_xgb) / 3

In [ ]:
submission_data = pd.DataFrame(data={"Id": test_ID, "SalePrice": pred_final})
submission_data.head()

In [ ]:
submission_data.to_csv("submission.csv", index=False)